# Notebook B — Tooth / cavity / crack segmentation (DentalAI)

Part of **Tooth-AI Photo Screening** (P2). Trains YOLOv8s-seg on DentalAI (tooth, caries, cavity,
crack polygons) so the photo report can attach findings to approximate tooth positions and
gate "is this a mouth?" on real tooth detections.

The Kaggle mirror is a Roboflow *semantic-mask* export (pixel values: 1 caries, 2 cavity,
3 crack, 4 tooth); it is converted to instance polygons, with touching teeth split by watershed.

**Run all** on Kaggle (add the dataset `pawanvalluri/dental-segmentation` as input, GPU T4) or
Colab (GPU; the dataset is fetched with `kagglehub`, falling back to Dataset Ninja). No interactive
cells. Budget ≈ 3 h on a T4 (training hard-capped at 2.5 h).

**Produces** `teeth_dentalai_artifacts.zip`: `best.pt`, `metrics.json`, `thresholds.json`,
`test_files.txt`, plots. Import with
`python photo_screening/scripts/fetch_weights.py --import-zip teeth_dentalai_artifacts.zip`.

**Data:** DentalAI (2,495 intraoral photos; tooth, caries, cavity, crack), **CC BY 4.0**
(datasetninja.com/dentalai; Kaggle mirror pawanvalluri/dental-segmentation).

In [ ]:
# 1. Environment
import os, sys, time, json, shutil, subprocess, platform, random, hashlib, zipfile, base64, io, zlib
from pathlib import Path

T0 = time.time()
def log(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("/content") if Path("/content").exists() else Path.cwd()
DATA = Path("/tmp/dentalai") if ON_KAGGLE else WORK / "dentalai"   # keep data out of the saved output
OUT = WORK / "teeth_dentalai_artifacts"
for p in (DATA, OUT, OUT / "plots"):
    p.mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3", "requests", "kagglehub"], check=True)
import numpy as np, requests, torch, ultralytics, cv2
from ultralytics import YOLO

SEED = 0
random.seed(SEED); np.random.seed(SEED)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
UNIFIED = ["tooth", "caries", "cavity", "crack"]                   # output class order (fixed)

def visible(p):                                           # skip macOS zip junk (__MACOSX/, ._AppleDouble files)
    return not p.name.startswith("._") and "__MACOSX" not in p.parts and p.name != ".DS_Store"
log(f"python {platform.python_version()} | torch {torch.__version__} | ultralytics {ultralytics.__version__} | "
    f"device {torch.cuda.get_device_name(0) if DEVICE == 0 else 'CPU'}")

In [ ]:
# 2. Helpers: resumable download with size check, box metrics
def download(url, dest, expected_size=None, tries=5):
    dest = Path(dest)
    for attempt in range(1, tries + 1):
        have = dest.stat().st_size if dest.exists() else 0
        if expected_size and have == expected_size:
            return dest
        headers = {"Range": f"bytes={have}-"} if have else {}
        try:
            with requests.get(url, headers=headers, stream=True, timeout=60) as r:
                if r.status_code == 416:                       # already complete
                    return dest
                r.raise_for_status()
                mode = "ab" if (have and r.status_code == 206) else "wb"
                total = int(r.headers.get("Content-Length", 0)) + (have if mode == "ab" else 0)
                done, last = (have if mode == "ab" else 0), time.time()
                with open(dest, mode) as f:
                    for chunk in r.iter_content(chunk_size=8 << 20):
                        f.write(chunk); done += len(chunk)
                        if time.time() - last > 20:
                            log(f"  {dest.name}: {done / 1e9:.2f} / {total / 1e9:.2f} GB"); last = time.time()
        except Exception as e:                                     # network hiccup -> resume
            log(f"  download attempt {attempt} failed: {e}")
            time.sleep(5 * attempt)
            continue
        if not expected_size or dest.stat().st_size == expected_size:
            return dest
        log(f"  size mismatch ({dest.stat().st_size} != {expected_size}), retrying")
    raise RuntimeError(f"download failed: {url}")


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def iou_matrix(a, b):
    """IoU between xyxy arrays a (N,4) and b (M,4)."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    tl = np.maximum(a[:, None, :2], b[None, :, :2]); br = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.prod(np.clip(br - tl, 0, None), axis=2)
    area = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)


def match(preds, gts, iou_thr=0.5):
    """preds: {img: (boxes, scores)}, gts: {img: boxes}. Returns (scores, is_tp, n_gt), greedy by score."""
    scores, tps, n_gt = [], [], 0
    for img, g in gts.items():
        n_gt += len(g)
        b, s = preds.get(img, (np.zeros((0, 4)), np.zeros(0)))
        order = np.argsort(-s); b, s = b[order], s[order]
        used = np.zeros(len(g), bool); ious = iou_matrix(b, g)
        for i in range(len(b)):
            j = int(np.argmax(ious[i])) if len(g) else -1
            ok = j >= 0 and ious[i, j] >= iou_thr and not used[j]
            if ok:
                used[j] = True
            scores.append(s[i]); tps.append(ok)
    return np.array(scores), np.array(tps, bool), n_gt


def pr_at(scores, tps, n_gt, thr):
    keep = scores >= thr
    tp = int(tps[keep].sum()); fp = int(keep.sum() - tp)
    p = tp / max(tp + fp, 1); r = tp / max(n_gt, 1)
    return {"conf": float(thr), "precision": p, "recall": r, "f1": 2 * p * r / max(p + r, 1e-9), "tp": tp, "fp": fp, "fn": n_gt - tp}


def ap50(scores, tps, n_gt):
    """101-point interpolated AP (same interpolation Ultralytics uses)."""
    if n_gt == 0 or len(scores) == 0:
        return 0.0
    o = np.argsort(-scores); tp = np.cumsum(tps[o]); fp = np.cumsum(~tps[o])
    rec = tp / n_gt; prec = tp / np.maximum(tp + fp, 1e-9)
    mrec = np.concatenate([[0.0], rec, [1.0]]); mpre = np.concatenate([[1.0], prec, [0.0]])
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    x = np.linspace(0, 1, 101)
    return float(np.trapezoid(np.interp(x, mrec, mpre), x) if hasattr(np, "trapezoid") else np.trapz(np.interp(x, mrec, mpre), x))


def image_level(preds, gts, thr):
    """'Does this image contain any caries?' sensitivity / specificity at a score threshold."""
    tp = fp = tn = fn = 0
    for img, g in gts.items():
        s = preds.get(img, (None, np.zeros(0)))[1]
        pos_pred, pos_true = bool((s >= thr).any()), len(g) > 0
        tp += pos_pred and pos_true; fn += (not pos_pred) and pos_true
        fp += pos_pred and not pos_true; tn += (not pos_pred) and not pos_true
    return {"conf": float(thr), "sensitivity": tp / max(tp + fn, 1), "specificity": tn / max(tn + fp, 1),
            "tp": tp, "fp": fp, "tn": tn, "fn": fn, "n_images": len(gts)}

log("helpers ready")

In [ ]:
# 3. Locate the dataset: Kaggle input -> kagglehub -> Dataset Ninja
def find_root():
    for base in [Path("/kaggle/input")] + ([Path(os.environ["DENTALAI_DIR"])] if os.environ.get("DENTALAI_DIR") else []):
        if base.exists():
            for p in sorted(filter(visible, base.rglob("*"))):
                if p.is_dir() and any(k in p.name.lower() for k in ("dental", "dentalai")):
                    return p
            if any(base.iterdir()):
                return base
    try:
        import kagglehub
        return Path(kagglehub.dataset_download("pawanvalluri/dental-segmentation"))
    except Exception as e:
        log(f"kagglehub unavailable ({e}); trying Dataset Ninja")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "dataset-tools"], check=True)
    import dataset_tools as dtools
    dtools.download(dataset="DentalAI", dst_dir=str(DATA / "ninja"))
    return DATA / "ninja"

ROOT = find_root()
log(f"dataset root: {ROOT}")
for p in sorted(filter(visible, ROOT.rglob("*"))):
    rel = p.relative_to(ROOT)
    if p.is_dir() and len(rel.parts) <= 3:
        print(f"{'  ' * (len(rel.parts) - 1)}{p.name}/  ({sum(1 for q in p.iterdir() if q.is_file() and visible(q))} files)")

In [ ]:
# 4. Detect the annotation format and convert to a YOLO-seg view with the unified class order
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
VIEW = DATA / "yolo_view"
SPLIT_ALIASES = {"train": ("train",), "valid": ("valid", "val", "validation"), "test": ("test",)}

def unify(name):
    n = str(name).lower()
    for u in UNIFIED:
        if u in n:
            return UNIFIED.index(u)
    return None

def split_of(path):
    parts = [x.lower() for x in path.parts]
    for s, al in SPLIT_ALIASES.items():
        if any(a in parts for a in al):
            return s
    return None

def write_label(split, img_path, polys):                 # polys: [(cls, [(x_norm, y_norm), ...])]
    (VIEW / split / "images").mkdir(parents=True, exist_ok=True); (VIEW / split / "labels").mkdir(parents=True, exist_ok=True)
    dst = VIEW / split / "images" / img_path.name
    if not dst.exists():
        dst.symlink_to(img_path.resolve())
    rows = [f"{c} " + " ".join(f"{x:.6f} {y:.6f}" for x, y in pts) for c, pts in polys if len(pts) >= 3]
    (VIEW / split / "labels" / f"{img_path.stem}.txt").write_text("\n".join(rows) + ("\n" if rows else ""))
    return len(rows)

def mask_to_polys(mask):
    cs, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return [c.reshape(-1, 2) for c in cs if len(c) >= 3]

counts = {}
def bump(split, n):
    c = counts.setdefault(split, {"images": 0, "objects": 0}); c["images"] += 1; c["objects"] += n

yaml_files = [p for p in ROOT.rglob("data.yaml") if visible(p)]
coco_files = [p for p in ROOT.rglob("*.json") if visible(p) and "coco" in p.name.lower() or p.name.startswith("_annotations")]
sly_meta = [p for p in ROOT.rglob("meta.json") if visible(p)]
mask_files = [p for p in ROOT.rglob("*_mask.png") if visible(p)][:1]
class_maps = {}

if yaml_files:
    FORMAT = "yolo-seg"
    import yaml
    d = yaml.safe_load(yaml_files[0].read_text()); src_names = d["names"]
    src_names = list(src_names.values()) if isinstance(src_names, dict) else list(src_names)
    remap = {i: unify(n) for i, n in enumerate(src_names)}
    for lab in filter(visible, ROOT.rglob("*.txt")):
        split = split_of(lab)
        if split is None or lab.parent.name != "labels":
            continue
        imgs = [q for q in (lab.parent.parent / "images").glob(lab.stem + ".*") if visible(q) and q.suffix.lower() in IMG_EXT]
        if not imgs:
            continue
        polys = []
        for line in lab.read_text(errors="replace").splitlines():
            v = line.split()
            if len(v) >= 7 and remap.get(int(v[0])) is not None:
                xy = list(map(float, v[1:])); polys.append((remap[int(v[0])], list(zip(xy[0::2], xy[1::2]))))
        bump(split, write_label(split, imgs[0], polys))
elif coco_files:
    FORMAT = "coco"
    for cj in coco_files:
        split = split_of(cj)
        if split is None:
            continue
        d = json.loads(cj.read_text()); cats = {c["id"]: unify(c["name"]) for c in d["categories"]}
        anns = {}
        for a in d["annotations"]:
            anns.setdefault(a["image_id"], []).append(a)
        for im in d["images"]:
            p = next(filter(visible, cj.parent.rglob(im["file_name"])), None)
            if p is None:
                continue
            w, h = im["width"], im["height"]; polys = []
            for a in anns.get(im["id"], []):
                c = cats.get(a["category_id"])
                if c is None or not isinstance(a.get("segmentation"), list):
                    continue
                for seg in a["segmentation"]:
                    polys.append((c, [(seg[i] / w, seg[i + 1] / h) for i in range(0, len(seg) - 1, 2)]))
            bump(split, write_label(split, p, polys))
elif sly_meta:
    FORMAT = "supervisely"
    for ann in filter(visible, ROOT.rglob("*.json")):
        if ann.parent.name != "ann":
            continue
        split = split_of(ann) or "train"
        img = ann.parent.parent / "img" / ann.name[:-5]             # "<image>.json" -> "<image>"
        if not img.exists():
            continue
        d = json.loads(ann.read_text()); w, h = d["size"]["width"], d["size"]["height"]; polys = []
        for o in d.get("objects", []):
            c = unify(o.get("classTitle", ""))
            if c is None:
                continue
            if o.get("geometryType") == "polygon":
                polys.append((c, [(x / w, y / h) for x, y in o["points"]["exterior"]]))
            elif o.get("geometryType") == "bitmap":
                raw = zlib.decompress(base64.b64decode(o["bitmap"]["data"]))
                m = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_UNCHANGED)
                m = (m[:, :, 3] if m.ndim == 3 and m.shape[2] == 4 else m) > 0
                ox, oy = o["bitmap"]["origin"]
                for c_pts in mask_to_polys(m):
                    polys.append((c, [((x + ox) / w, (y + oy) / h) for x, y in c_pts]))
        bump(split, write_label(split, img, polys))
elif mask_files:
    # Roboflow "PNG masks for semantic segmentation" (the Kaggle mirror): <img>.jpg + <img>_mask.png,
    # pixel values mapped by <split>/_classes.csv. Semantic masks merge touching objects, so tooth
    # blobs are split with a distance-transform watershed; findings use connected components.
    FORMAT = "semantic-masks"
    MAX_SIDE = 1280                                        # polygons are normalised; work at reduced size

    def class_map(split_dir):
        csv_path = split_dir / "_classes.csv"
        if not csv_path.exists():
            return {1: 1, 2: 2, 3: 3, 4: 0}                # DentalAI default: Caries, Cavity, Crack, Tooth
        out = {}
        for line in csv_path.read_text(errors="replace").splitlines()[1:]:
            v, _, name = line.partition(",")
            if v.strip().isdigit() and unify(name.strip()) is not None:
                out[int(v)] = unify(name.strip())
        return out

    def split_touching(binary):
        """Split touching teeth: one seed per distance-transform core, then watershed. Labels:
        1 = sure background, 2.. = seeds, 0 = undecided foreground."""
        dist = cv2.distanceTransform(binary, cv2.DIST_L2, 5)
        n, comps = cv2.connectedComponents(binary)
        markers = np.zeros(binary.shape, np.int32)
        next_id = 2
        for k in range(1, n):
            blob = comps == k
            m, cores = cv2.connectedComponents(((dist > 0.5 * dist[blob].max()) & blob).astype(np.uint8))
            for j in range(1, m):
                markers[cores == j] = next_id
                next_id += 1
        markers[cv2.dilate(binary, np.ones((3, 3), np.uint8)) == 0] = 1
        cv2.watershed(cv2.cvtColor(binary * 255, cv2.COLOR_GRAY2BGR), markers)
        return [((markers == k) & (binary > 0)).astype(np.uint8) for k in range(2, next_id)]

    for mpath in sorted(filter(visible, ROOT.rglob("*_mask.png"))):
        split = split_of(mpath)
        img = next((q for q in mpath.parent.glob(mpath.name[:-len("_mask.png")] + ".*")
                    if visible(q) and q.suffix.lower() in IMG_EXT), None)
        if split is None or img is None:
            continue
        cmap = class_maps.setdefault(mpath.parent, class_map(mpath.parent))
        mask = cv2.imread(str(mpath), cv2.IMREAD_UNCHANGED)
        mask = mask[:, :, 0] if mask.ndim == 3 else mask
        h0, w0 = mask.shape
        s = min(1.0, MAX_SIDE / max(h0, w0))
        if s < 1:
            mask = cv2.resize(mask, (round(w0 * s), round(h0 * s)), interpolation=cv2.INTER_NEAREST)
        h, w = mask.shape; min_area = 0.0002 * h * w; polys = []
        for value, c in cmap.items():
            binary = (mask == value).astype(np.uint8)
            if not binary.any():
                continue
            if c == 0:
                parts = split_touching(binary)
            else:
                n, lab = cv2.connectedComponents(binary)
                parts = [(lab == k).astype(np.uint8) for k in range(1, n)]
            for part in parts:
                if part.sum() < min_area:
                    continue
                cs, _ = cv2.findContours(part, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                cnt = max(cs, key=cv2.contourArea)
                cnt = cv2.approxPolyDP(cnt, 0.002 * cv2.arcLength(cnt, True), True).reshape(-1, 2)
                if len(cnt) >= 3:
                    polys.append((c, [(x / w, y / h) for x, y in cnt]))
        bump(split, write_label(split, img, polys))
else:
    raise SystemExit("Unrecognised dataset format (no data.yaml, COCO json, Supervisely meta.json or *_mask.png) — inspect the tree above.")

log(f"format: {FORMAT} | {counts}")
if "valid" not in counts or "test" not in counts:
    # Carve a fixed, seeded split from train if the export lacks one
    imgs = sorted(filter(visible, (VIEW / "train" / "images").iterdir())); random.Random(SEED).shuffle(imgs)
    n = max(1, len(imgs) // 10)
    for split, chunk in (("test", imgs[:n]), ("valid", imgs[n:2 * n])):
        if split in counts:
            continue
        for p in chunk:
            for sub, ext in (("images", p.suffix), ("labels", ".txt")):
                (VIEW / split / sub).mkdir(parents=True, exist_ok=True)
                shutil.move(str(VIEW / "train" / sub / (p.stem + ext)), VIEW / split / sub / (p.stem + ext))
        log(f"carved seeded {split} split: {len(chunk)} images")
DATA_YAML = DATA / "data.yaml"
DATA_YAML.write_text(json.dumps({"path": str(VIEW), "train": "train/images", "val": "valid/images",
                                 "test": "test/images", "names": UNIFIED}))
test_files = sorted(p.name for p in (VIEW / "test" / "images").iterdir() if visible(p))
(OUT / "test_files.txt").write_text("\n".join(test_files) + "\n")
log(f"test split = {len(test_files)} images")

In [ ]:
# 5. Train YOLOv8s-seg
model = YOLO("yolov8s-seg.pt")
model.train(data=str(DATA_YAML), imgsz=640, epochs=50, patience=15, batch=-1, seed=SEED, deterministic=True,
            device=DEVICE, workers=4, time=2.5, project=str(WORK / "runs"), name="teeth_dentalai",
            exist_ok=True, plots=True, verbose=False)
RUN = Path(model.trainer.save_dir); BEST = RUN / "weights" / "best.pt"
log(f"training done -> {BEST}")

In [ ]:
# 6. Per-class predictions on valid/test (boxes, conf 0.001) for calibration
def load_gts(split, w_h):
    out = {}
    for img in sorted(filter(visible, (VIEW / split / "images").iterdir())):
        w, h = w_h[img.name]; boxes = {c: [] for c in range(len(UNIFIED))}
        for line in (VIEW / split / "labels" / f"{img.stem}.txt").read_text().splitlines():
            v = line.split()
            if len(v) < 7:
                continue
            xy = np.array(list(map(float, v[1:]))).reshape(-1, 2) * [w, h]
            boxes[int(v[0])].append([*xy.min(0), *xy.max(0)])
        out[img.name] = {c: np.array(b).reshape(-1, 4) for c, b in boxes.items()}
    return out

def predict_split(split):
    m = YOLO(str(BEST)); preds, sizes = {}, {}
    files = sorted(filter(visible, (VIEW / split / "images").iterdir()))
    for i in range(0, len(files), 32):
        for r in m.predict([str(f) for f in files[i:i + 32]], imgsz=640, conf=0.001, iou=0.6, device=DEVICE,
                           verbose=False, max_det=300):
            name = Path(r.path).name; h, w = r.orig_shape; sizes[name] = (w, h)
            cls = r.boxes.cls.cpu().numpy().astype(int); xyxy = r.boxes.xyxy.cpu().numpy(); conf = r.boxes.conf.cpu().numpy()
            preds[name] = {c: (xyxy[cls == c], conf[cls == c]) for c in range(len(UNIFIED))}
    return preds, load_gts(split, sizes)

per = lambda d, c: {k: v[c] for k, v in d.items()}
val_p, val_g = predict_split("valid"); test_p, test_g = predict_split("test")
grid = np.round(np.arange(0.01, 0.96, 0.01), 2)
thresholds = {"model_id": "teeth_dentalai", "classes": {}, "method": "valid split, box-level IoU>=0.5; "
              "t_low = highest conf with recall>=0.85; t_med = max-F1; t_high = lowest conf above t_med with precision>=0.80"}
curves = {}
for c, name in enumerate(UNIFIED):
    s, tp, n = match(per(val_p, c), per(val_g, c))
    if n == 0:
        continue
    curve = [pr_at(s, tp, n, t) for t in grid]
    curves[name] = curve
    ok90 = [x["conf"] for x in curve if x["recall"] >= 0.90]
    best_f1 = max(curve, key=lambda x: x["f1"])["conf"]
    t_low = max(ok90) if ok90 else None                   # None: recall target unreachable (weak class)
    t_med = max(t_low or 0.0, best_f1)                    # same rule as scripts/calibrate.py
    hi = [x["conf"] for x in curve if x["precision"] >= 0.90 and x["conf"] >= t_med]
    thresholds["classes"][name] = {"t_low": t_low if t_low is not None else t_med, "t_med": t_med,
                                   "t_high": min(hi) if hi else round(min(t_med + 0.10, 0.95), 2),
                                   "recall_target_reached": t_low is not None}
thresholds["method"] = ("valid split, box-level IoU>=0.5; t_low = highest conf with recall>=0.90; "
                        "t_med = max-F1 conf (>= t_low); t_high = lowest conf >= t_med with precision>=0.90")
thresholds["valid_curves"] = curves
(OUT / "thresholds.json").write_text(json.dumps(thresholds, indent=2))
log(f"thresholds: { {k: {kk: vv for kk, vv in v.items()} for k, v in thresholds['classes'].items()} }")

In [ ]:
# 7. Test split metrics (Ultralytics box + mask) and image-level presence per finding class
val = YOLO(str(BEST)).val(data=str(DATA_YAML), split="test", imgsz=640, batch=16, conf=0.001, iou=0.6, device=DEVICE,
                          plots=True, project=str(WORK / "runs"), name="teeth_dentalai_test", exist_ok=True, verbose=False)
per_class = {}
for i, c in enumerate(val.box.ap_class_index):
    per_class[UNIFIED[int(c)]] = {"box_ap50": float(val.box.ap50[i]), "mask_ap50": float(val.seg.ap50[i])}
own = {}
for c, name in enumerate(UNIFIED):
    s, tp, n = match(per(test_p, c), per(test_g, c))
    t = thresholds["classes"].get(name, {}).get("t_low", 0.25)
    own[name] = {"ap50": ap50(s, tp, n), "n_boxes": int(n), "at_t_low": pr_at(s, tp, n, t),
                 "image_level_at_t_low": image_level({k: v[c] for k, v in test_p.items()}, per(test_g, c), t)}
for f in ("BoxPR_curve.png", "MaskPR_curve.png", "confusion_matrix.png", "confusion_matrix_normalized.png"):
    if (Path(val.save_dir) / f).exists():
        shutil.copy(Path(val.save_dir) / f, OUT / "plots" / f"test_{f}")
if (RUN / "results.png").exists():
    shutil.copy(RUN / "results.png", OUT / "plots" / "results.png")
shutil.copy(BEST, OUT / "best.pt")
metrics = {"model_id": "teeth_dentalai", "arch": "yolov8s-seg", "imgsz": 640, "seed": SEED, "classes": UNIFIED,
           "best_pt_sha256": sha256(OUT / "best.pt"), "source_format": FORMAT, "splits": counts,
           "tooth_instances": "approximate: semantic masks split by watershed" if FORMAT == "semantic-masks" else "annotated instances",
           "test": {"box_map50": float(val.box.map50), "mask_map50": float(val.seg.map50),
                    "box_map50_95": float(val.box.map), "mask_map50_95": float(val.seg.map),
                    "per_class": per_class, "own_matching": own},
           "thresholds": thresholds["classes"],
           "dataset": {"name": "DentalAI", "licence": "CC-BY-4.0", "source": "datasetninja.com/dentalai; kaggle pawanvalluri/dental-segmentation"},
           "env": {"torch": torch.__version__, "ultralytics": ultralytics.__version__,
                   "gpu": torch.cuda.get_device_name(0) if DEVICE == 0 else "cpu"},
           "wall_minutes": round((time.time() - T0) / 60, 1)}
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=2))
zip_path = shutil.make_archive(str(WORK / "teeth_dentalai_artifacts"), "zip", OUT)
log(f"DONE -> {zip_path}")
print(json.dumps(metrics["test"], indent=2)[:3000])